# 使用 Unsloth 对 DeepSeek-R1-Distill-Qwen-1.5B 模型进行 LoRA 微调

本 Notebook 展示了如何使用 `unsloth` 库对 `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` 模型进行高效的 QLoRA (Low-Rank Adaptation) 微调。

整个流程包括：
1.  环境准备与库导入
2.  加载预训练模型和分词器 (Tokenizer)。
3.  在微调前，对模型进行简单的推理测试。
4.  下载和格式化训练数据集
5.  使用 `unsloth` 的 `FastLanguageModel` 来为模型添加 LoRA 适配器。
6.  配置 `SFTTrainer` 监督微调训练配置。
7.  启动训练，并观察 Loss 变化情况
8.  保存微调后的模型
9.  测试训练后的生成结果

### 1. 环境准备与库导入

首先，我们需要安装并导入所有必要的库。`transformers` 用于加载模型和分词器，`unsloth` 用于高效微调，`trl` 提供了 `SFTTrainer`，而 `datasets` 用于处理数据。

**注意**: 在运行此 Notebook 之前，请确保已安装所有依赖包：
```bash
pip install -r requirements.txt
datasets
pandas
peft
timm
torch
torchvision
transformers
trl
unsloth
unsloth_zoo
```

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig, DataCollatorForSeq2Seq
from datasets import Dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### 2. 加载预训练模型和分词器 (Tokenizer)

In [2]:
# 定义模型和一些基本参数
max_seq_length = 2048
dtype = None # None 表示自动选择 (Float16 a T4, V100, BFloat16 a Ampere)
load_in_4bit = True # 使用 4bit 量化加载

# 这是您的模型标识符，请替换为您正在使用的模型
# 例如："qwen-1.5b_lora_model"
# model_name = "qwen-1.5b_lora_model" 
# model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B" 
model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit" 

# 这一步会返回一个经过 Unsloth 优化的模型和一个分词器
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


==((====))==  Unsloth 2025.8.5: Fast Qwen2 patching. Transformers: 4.55.2.
   \\   /|    NVIDIA A10. Num GPUs = 1. Max memory: 23.479 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


### 3. 微调前推理测试

在对模型进行任何修改之前，我们先用它来生成一段文本，看看原始模型的表现如何。这可以作为我们微调效果的基准参考。

In [3]:
# 模型推理的 Prompt 模板
inference_prompt = """以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
{}

### Response:
<think>{}
"""

In [4]:
FastLanguageModel.for_inference(model)

question = "男，28岁，程序员，最近一周每天工作到半夜，感觉头晕、脖子疼，有时候还恶心。"

inputs = tokenizer([inference_prompt.format(question, "")], return_tensors="pt").to("cuda")
attention_mask = inputs.input_ids.ne(tokenizer.pad_token_id).long().to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)

In [5]:
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [6]:
print(response[0].split("### Response:")[1])


<think>
好的，我现在需要帮助用户分析一个28岁的男性程序员的问题。用户最近工作到半夜，感觉头晕、脖子疼和恶心，这可能与长时间工作、神经损伤或脑损伤有关。

首先，我会考虑是否需要进行神经损伤检查，因为头晕和恶心可能提示有神经损伤。如果检查结果显示有脑损伤，可能需要进行脑电图或MRI以评估情况。如果没问题，可能需要进一步评估是否有其他损伤，比如脊髓损伤或脊柱问题。

然后，我会思考是否有其他因素，比如长时间工作是否影响脑部结构，导致晕头转向。可能需要咨询神经科医生或脑科医生，看看是否有长期的脑损伤或神经损伤。

接下来，我需要考虑是否有其他可能的健康问题，比如中风、脑血管问题或神经系统疾病。如果发现有其他风险因素，可能需要进行相关检查，如心电图、血常规、血糖监测等。

此外，用户可能需要评估是否有其他潜在的健康问题，比如是否长期使用过强或过弱的药物，或者是否存在其他神经系统疾病如多发性硬化症或阿尔茨海默病的风险。

如果这些检查没有问题，可能需要进行神经科或脑科的进一步治疗，比如药物治疗、手术或生活方式干预。如果检查结果有误，可能需要重新评估或调整治疗方案。

最后，我会建议用户尽快进行专业检查，确保没有其他潜在的健康问题，并根据检查结果制定个性化的治疗计划。
</think>

男，28岁，程序员，最近一周每天工作到半夜，感觉头晕、脖子疼，有时候还恶心。这可能与长时间工作、神经损伤或脑损伤有关。建议尽快进行神经损伤或脑损伤检查，评估是否有神经损伤或脑损伤。如果检查结果显示有脑损伤，可能需要进行脑电图或MRI以评估情况。如果没问题，可能需要进一步评估是否有其他损伤，如脊髓损伤或脊柱问题。如果检查结果有误，建议重新评估或调整治疗方案。尽快进行专业检查，确保没有其他潜在的健康问题，并根据检查结果制定个性化的治疗计划。


---

### 4. 下载和格式化训练数据集


医学推理数据集：https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/viewer/zh

![dataset](images/dataset.png)

In [7]:
# 模型训练的 Prompt 模板
train_prompt = """以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
{}

### Response:
<think>
{}
</think>
{}
"""

In [8]:
EOS_TOKEN = tokenizer.eos_token # 添加 EOS Token

def formatting_prompts_func(examples):
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        # 将 EOS Token 添加到样本最后
        text = train_prompt.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "zh", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [9]:
dataset[0]["text"]

'以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。\n请撰写一份恰当的回复，以完成该请求。\n在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。\n\n\n### Instruction:\n你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。\n请回答以下医学问题。\n\n### Question:\n根据描述，一个1岁的孩子在夏季头皮出现多处小结节，长期不愈合，且现在疮大如梅，溃破流脓，口不收敛，头皮下有空洞，患处皮肤增厚。这种病症在中医中诊断为什么病？\n\n### Response:\n<think>\n这个小孩子在夏天头皮上长了些小结节，一直都没好，后来变成了脓包，流了好多脓。想想夏天那么热，可能和湿热有关。才一岁的小孩，免疫力本来就不强，夏天的湿热没准就侵袭了身体。\n\n用中医的角度来看，出现小结节、再加上长期不愈合，这些症状让我想到了头疮。小孩子最容易得这些皮肤病，主要因为湿热在体表郁结。\n\n但再看看，头皮下还有空洞，这可能不止是简单的头疮。看起来病情挺严重的，也许是脓肿没治好。这样的情况中医中有时候叫做禿疮或者湿疮，也可能是另一种情况。\n\n等一下，头皮上的空洞和皮肤增厚更像是疾病已经深入到头皮下，这是不是说明有可能是流注或瘰疬？这些名字常描述头部或颈部的严重感染，特别是有化脓不愈合，又形成通道或空洞的情况。\n\n仔细想想，我怎么感觉这些症状更贴近瘰疬的表现？尤其考虑到孩子的年纪和夏天发生的季节性因素，湿热可能是主因，但可能也有火毒或者痰湿造成的滞留。\n\n回到基本的症状描述上看，这种长期不愈合又复杂的状况，如果结合中医更偏重的病名，是不是有可能是涉及更深层次的感染？\n\n再考虑一下，这应该不是单纯的瘰疬，得仔细分析头皮增厚并出现空洞这样的严重症状。中医里头，这样的表现可能更符合‘蚀疮’或‘头疽’。这些病名通常描述头部严重感染后的溃烂和组织坏死。\n\n看看季节和孩子的体质，夏天又湿又热，外邪很容易侵入头部，对孩子这么弱的免疫系统简直就是挑战。头疽这个病名听起来真是切合，因为它描述的感染严重，溃烂到出现空洞。\n\n不过，仔细琢磨后发现，还有个病名似乎更为合适，叫做‘蝼蛄疖’，这病在中医里专指像这种严重感染并伴有深部空洞的情况。它也涵盖了化脓和皮肤增厚这些症状。\n\n

In [10]:
from IPython.display import display, Markdown

display(Markdown(dataset[0]["text"]))

以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
根据描述，一个1岁的孩子在夏季头皮出现多处小结节，长期不愈合，且现在疮大如梅，溃破流脓，口不收敛，头皮下有空洞，患处皮肤增厚。这种病症在中医中诊断为什么病？

### Response:
<think>
这个小孩子在夏天头皮上长了些小结节，一直都没好，后来变成了脓包，流了好多脓。想想夏天那么热，可能和湿热有关。才一岁的小孩，免疫力本来就不强，夏天的湿热没准就侵袭了身体。

用中医的角度来看，出现小结节、再加上长期不愈合，这些症状让我想到了头疮。小孩子最容易得这些皮肤病，主要因为湿热在体表郁结。

但再看看，头皮下还有空洞，这可能不止是简单的头疮。看起来病情挺严重的，也许是脓肿没治好。这样的情况中医中有时候叫做禿疮或者湿疮，也可能是另一种情况。

等一下，头皮上的空洞和皮肤增厚更像是疾病已经深入到头皮下，这是不是说明有可能是流注或瘰疬？这些名字常描述头部或颈部的严重感染，特别是有化脓不愈合，又形成通道或空洞的情况。

仔细想想，我怎么感觉这些症状更贴近瘰疬的表现？尤其考虑到孩子的年纪和夏天发生的季节性因素，湿热可能是主因，但可能也有火毒或者痰湿造成的滞留。

回到基本的症状描述上看，这种长期不愈合又复杂的状况，如果结合中医更偏重的病名，是不是有可能是涉及更深层次的感染？

再考虑一下，这应该不是单纯的瘰疬，得仔细分析头皮增厚并出现空洞这样的严重症状。中医里头，这样的表现可能更符合‘蚀疮’或‘头疽’。这些病名通常描述头部严重感染后的溃烂和组织坏死。

看看季节和孩子的体质，夏天又湿又热，外邪很容易侵入头部，对孩子这么弱的免疫系统简直就是挑战。头疽这个病名听起来真是切合，因为它描述的感染严重，溃烂到出现空洞。

不过，仔细琢磨后发现，还有个病名似乎更为合适，叫做‘蝼蛄疖’，这病在中医里专指像这种严重感染并伴有深部空洞的情况。它也涵盖了化脓和皮肤增厚这些症状。

哦，该不会是夏季湿热，导致湿毒入侵，孩子的体质不能御，其病情发展成这样的感染？综合分析后我觉得‘蝼蛄疖’这个病名真是相当符合。
</think>
从中医的角度来看，你所描述的症状符合“蝼蛄疖”的病症。这种病症通常发生在头皮，表现为多处结节，溃破流脓，形成空洞，患处皮肤增厚且长期不愈合。湿热较重的夏季更容易导致这种病症的发展，特别是在免疫力较弱的儿童身上。建议结合中医的清热解毒、祛湿消肿的治疗方法进行处理，并配合专业的医疗建议进行详细诊断和治疗。
<｜end▁of▁sentence｜>

### 5. 使用 Unsloth 添加 LoRA 适配器

这是使用 `unsloth` 的核心步骤。我们调用 `FastLanguageModel.get_peft_model`，它会非常高效地为模型注入 LoRA 模块。

- `r`: LoRA 的秩 (rank)，是控制模型复杂度和参数量的关键超参数。
- `target_modules`: 指定要在哪些线性层（如注意力机制的 q, k, v, o 投影层）上应用 LoRA。
- `lora_alpha`: LoRA 的缩放因子，通常设置为 `r` 的两倍或与 `r` 相同。
- `use_gradient_checkpointing`: 一种节省显存的技术，对于训练大模型至关重要。

In [11]:
# 因为 `model` 对象现在是由 Unsloth 创建的，它包含了所有必需的属性
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=1432,
    use_rslora=False,
    loftq_config=None,
)
# 检查模型结构，确认 LoRA 适配器已添加
print(model)

Unsloth 2025.8.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
       

### 6. 配置 SFTTrainer

`SFTTrainer` (Supervised Fine-tuning Trainer) 是一个专门用于指令微调的训练器。我们需要配置 `TrainingArguments` 来指定所有的训练参数，如批量大小、学习率、优化器等。

In [12]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        # max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 1432,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
        bf16=True,
        fp16=False
    ),
)

[2026-04-15 01:00:37,111] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/tiger/miniforge3/envs/py310/lib/python3.10/site-packages/deepspeed/runtime/zero/linear.py:49: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, weight, bias=None):
/home/tiger/miniforge3/envs/py310/lib/python3.10/site-packages/deepspeed/runtime/zero/linear.py:67: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):


### 7. 开始训练

一切准备就绪后，调用 `trainer.train()` 即可开始微调过程。训练结束后，会返回包含训练统计信息（如训练损失）的对象。

In [13]:
trainer_stats = trainer.train()

# 打印训练统计信息
print(trainer_stats)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,171 | Num Epochs = 1 | Total steps = 1,261
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,795,552,768 (1.03% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.214400
2,3.258300
3,3.198700
4,3.195600
5,3.163500
6,2.962300
7,2.974800
8,2.930400
9,2.896600
10,2.764400


TrainOutput(global_step=1261, training_loss=2.072256960157928, metrics={'train_runtime': 3742.9547, 'train_samples_per_second': 5.389, 'train_steps_per_second': 0.337, 'total_flos': 1.343852267960494e+17, 'train_loss': 2.072256960157928})


### 8. 保存微调后的模型（Lora）

训练完成后，您可以再次进行推理，比较微调后的模型与原始模型的差异。如果对结果满意，可以使用 `model.save_pretrained("your_lora_adapter_path")` 来保存训练好的 LoRA 适配器。

In [14]:
# model_dir_path="qwen-1.5b_lora_model-mini_60_steps"
model_dir_path="../models/qwen-1.5b_lora_model"
model.save_pretrained(model_dir_path)

In [15]:
tokenizer.save_pretrained(model_dir_path)

('../models/qwen-1.5b_lora_model/tokenizer_config.json',
 '../models/qwen-1.5b_lora_model/special_tokens_map.json',
 '../models/qwen-1.5b_lora_model/chat_template.jinja',
 '../models/qwen-1.5b_lora_model/tokenizer.json')

In [16]:
# 模型保存方式二选一（要么使用上面的分开保存，要么使用这里的合并 Lora 保存）
# model.save_pretrained_merged("qwen-1.5b_lora_model", tokenizer, save_method="merged_16bit")

### 9. 测试训练后的生成结果

In [17]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

question="一个患有急性阑尾炎的病人已经发病5天，腹痛稍有减轻但仍然发热，在体检时发现右下腹有压痛的包块，此时应如何处理？", # Question
inputs = tokenizer([inference_prompt.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1000,
    use_cache=True,
)

In [18]:
output = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(output[0].split("### Response:")[1])


<think>
这个病人有急性阑尾炎，已经5天了，嗯，现在腹痛稍微减轻了，但还是有点发热。哎，这种情况让我有点担心。首先，我得看看右下腹有没有压痛的包块。如果包块没有，那就不用太担心了，但如果有，那就要小心了。

嗯，这种情况下，我想应该先试试用抗生素来试试看，比如头孢曲松或者头孢曲松联合青霉素。这样做可以给病人提供一些抗生素的控制，看看效果如何。

不过，也不能太草率，因为抗生素虽然能对抗感染，但有时候可能对包块有影响。所以，如果包块存在，我得考虑一下是否需要手术来处理。

哦，对了，包块的存在可能意味着阑尾炎已经进展到更严重的状态，可能需要手术。手术不仅能直接解决问题，还能防止感染扩散。当然，手术时需要医生和麻醉师的帮助，确保安全。

所以，总结一下，如果右下腹有压痛的包块，最好先用抗生素试试看。如果包块没有，继续用药。如果包块有，那就得考虑手术了。这样，我们就能确保病人能得到合适的治疗和控制。
</think>
在处理病人急性阑尾炎的情况下，如果右下腹有压痛的包块，建议首先尝试使用抗生素治疗。头孢曲松和头孢曲松联合青霉素是常用的抗生素，可以有效地控制感染。如果经过抗生素治疗后症状有所缓解，那么可以考虑手术来处理阑尾炎的进展。手术不仅能直接解决问题，还能防止感染的扩散，因此在有包块的情况下，手术是合理的选择。如果包块不存在，继续使用抗生素治疗即可。总之，通过综合评估和选择合适的治疗方案，可以有效地控制阑尾炎的病情。



In [19]:
def generate_response(question: str, model, tokenizer, inference_prompt: str, max_new_tokens: int = 1024) -> str:
    """
    使用指定的模型和分词器为给定的医学问题生成响应。

    Args:
        question (str): 需要模型回答的医学问题。
        model: 已加载的 Unsloth/Hugging Face 模型。
        tokenizer: 对应的分词器。
        inference_prompt (str): 用于格式化输入的 f-string 模板。
        max_new_tokens (int, optional): 生成响应的最大 token 数量。默认为 1024。

    Returns:
        str: 模型生成的响应文本，已去除 prompt 部分。
    """
    # 1. 使用模板格式化输入
    prompt = inference_prompt.format(
        question, # 填充问题
        "",       # 留空，让模型生成 CoT 和 Response
    )

    # 2. 将格式化后的 prompt 进行分词，并转移到 GPU
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    # 3. 使用模型生成输出
    # use_cache=True 用于加速解码过程
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        use_cache=True,
    )
    
    # 4. 将生成的 token 解码为文本
    # skip_special_tokens=True 会移除像 EOS_TOKEN 这样的特殊标记
    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # 5. 切分字符串，只返回 "### Response:" 之后的部分
    # 使用 .split() 分割并获取响应内容，.strip() 用于去除可能存在的前后空白字符
    response_part = decoded_output.split("### Response:")
    if len(response_part) > 1:
        return response_part[1].strip()
    else:
        # 如果模型没有生成 "### Response:" 标记，则返回整个生成内容以供调试
        return decoded_output

In [20]:
my_question = "对于一名60岁男性患者，出现右侧胸疼并在X线检查中显示右侧肋膈角消失，诊断为肺结核伴右侧胸腔积液，请问哪一项实验室检查对了解胸水的性质更有帮助？"

response = generate_response(my_question, model, tokenizer, inference_prompt)
print("==================== 模型回答 ====================")
print(response)

==================== 模型回答 ====================
<think>
这个60岁的男性患者，右侧胸疼，X线显示右侧肋膈角消失，这让我首先想到可能和肺结核有关。肺结核通常会导致胸腔积液，所以这个症状很符合。现在，我需要弄清楚胸水的性质。胸水的性质非常重要，因为它会影响治疗方案和后续的诊断。

首先，胸水的酸碱度是关键。酸碱度的变化可能会影响肺部的反应，比如增加胸痛或者加重感染。因此，我应该检查胸水的酸碱度。这个检测结果能告诉我胸水是酸性还是碱性，这对后续的治疗非常重要。

然后，胸水的成分也很重要。像蛋白质、脂质、水和电解质这些都可能影响胸水的性质。比如，如果胸水里有大量蛋白质，这可能会让胸水变得更酸，或者更难分解。因此，分析胸水中的蛋白质含量和脂质的成分是很有帮助的。

此外，胸水的酸碱性也会影响胸腔的反应。酸性胸水可能增加胸痛，而碱性胸水则可能导致胸膜受压。所以，了解酸碱性水平也能给我一些信息。

总的来说，为了了解胸水的性质，我需要做几个关键的实验室检查。首先，酸碱度检测是个必须的步骤，因为它直接决定了胸水的酸碱性。然后，我还需要分析胸水中的蛋白质和脂质含量，这有助于了解胸水的成分，从而更好地预测胸腔反应。

嗯，看来这些检查能给我全面的了解。通过这些检测，我能够更好地指导后续的治疗方案，确保胸水的管理是合理的。
</think>
对于这位60岁男性患者，他的症状和X线检查结果提示可能存在肺结核伴胸腔积液。为了了解胸水的性质，几个关键的实验室检查应被考虑：

1. **胸水的酸碱度检测**：通过酸碱度检测，可以判断胸水是酸性还是碱性，这对后续的胸腔反应和治疗方案的制定非常重要。酸性胸水可能增加胸痛，而碱性胸水则可能导致胸膜受压。

2. **胸水成分分析**：检查胸水中的蛋白质和脂质含量可以帮助评估胸水的成分，从而更好地预测胸腔反应。蛋白质和脂质的含量变化可能影响胸水的酸碱性，进而影响胸腔反应。

3. **胸水的酸碱性水平**：通过分析酸碱性，可以进一步了解胸水的性质，从而指导胸腔的反应管理。

通过这些检查，可以全面了解胸水的性质，为后续的治疗方案制定提供依据。


In [21]:
my_question = "对于一名 28 岁的男性患者，工作是程序员，常年熬夜，最近突然感觉头晕目眩，甚至有点恶心。请问有可能是什么疾病？"

response = generate_response(my_question, model, tokenizer, inference_prompt)
print("==================== 模型回答 ====================")
print(response)

==================== 模型回答 ====================
<think>
这个28岁的男性患者，平时工作是程序员，这让我想到他可能有些身体上的问题。他最近开始感到头晕目眩，还有点恶心，这让我有点担心。

首先，他工作时间长，常常熬夜，这会不会是某种身体的代谢问题呢？比如说，过度工作可能导致身体的某些代谢产物被过度利用，从而引发一些不适。

嗯，头晕目眩和恶心，这让我想到了几种常见的代谢性疾病。比如，肝病或者肝硬化，这些都会影响到肝脏，导致头晕和恶心。不过，肝病一般需要更深入的检查，而肝硬化通常也会有其他症状，比如尿素氮和肌酐水平的升高。

不过，再想想，他最近工作时间长，可能更可能有与工作相关的代谢问题。他工作时间长，可能意味着他的大脑和脑干的代谢活动增加，从而导致一些代谢问题。

另外，他工作时间长，常常熬夜，也可能是体内代谢活动的过度活跃，这可能对肝脏产生影响，比如肝功能不全。这让我想到，肝脏问题，比如肝硬化，虽然不太常见，但也有可能。

不过，也不能仅仅局限于肝脏。他现在头晕和恶心，还可能涉及到其他类型的代谢性问题，比如代谢性酸中毒，或者代谢性低血糖，这些都是常见的代谢性疾病的表现。

嗯，代谢性酸中毒和代谢性低血糖这两种疾病，都可能引起头晕和恶心，尤其是在长时间的代谢活动下。而他工作时间长，可能让他更容易出现这些症状。

所以，结合他的工作背景和症状表现，我倾向于认为他可能患有代谢性酸中毒，比如代谢性酸中毒症，这种疾病常常伴随头晕、恶心和低血糖。

不过，我还需要进一步确认，因为代谢性酸中毒可能与他当前的状况更为契合。但，我必须考虑到他可能有其他潜在的代谢性疾病，比如肝病，也可能是代谢性低血糖。

综合来看，我倾向于认为他可能有代谢性酸中毒，但这不能排除其他潜在的代谢性疾病，比如肝病或肝硬化，因为这些也可能导致头晕和恶心。

最终，我觉得最可能的诊断是代谢性酸中毒，但也要考虑其他可能性。总之，他可能需要进一步的检查，以确认他的具体健康状况。
</think>
根据您提供的信息，这名28岁的男性患者可能患有代谢性酸中毒，这与他长时间的熬夜和头晕目眩的症状相符。代谢性酸中毒通常表现为头晕、恶心、低血糖等，而他工作时间长，可能让他更容易出现这些症状。因此，建议进行进一步的检查，如血液检查以确定酸中毒的程度，以排除其他可能的代谢性